# 🧠 Analisis BCI: Neurandiar
**Penelitian:** Dekoding *Imagined Speech* Bahasa Indonesia menggunakan Emotiv EPOC X
**Metode:** EEGNet, Logistic Regression, ML Klasik, dan Optuna Hyperparameter Tuning

Notebook ini bertindak sebagai **Master Journal**. Berfungsi untuk menarik, membersihkan, membandingkan, dan memvisualisasikan data dari 11 skenario eksperimen yang telah dieksekusi oleh sistem arsitektur backend.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from IPython.display import Image, display

# Konfigurasi Estetika Grafik Jurnal (Standar Publikasi)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    'font.size': 12, 
    'figure.dpi': 150, 
    'axes.titlesize': 14, 
    'axes.labelsize': 12,
    'font.family': 'sans-serif'
})

# Koneksi ke Database MLflow
# Sesuaikan jika Anda memindahkan lokasi notebook
MLFLOW_DB_PATH = os.path.abspath(os.path.join("..", "backend", "dataset", "logs", "mlflow", "mlruns.db"))
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")

print(f"✅ Sistem Telemetri Aktif. Terhubung ke: {MLFLOW_DB_PATH}")

## 1. Ekstraksi Metrik Seluruh Eksperimen (E0 - E10)
Menarik metrik akurasi validasi terbaik dari Model Deep Learning (Tingkat Suku Kata) dan Model Logistic Regression (Tingkat Kata Utuh).

In [ ]:
# Ambil semua data eksperimen dari MLflow
all_experiments = [exp.experiment_id for exp in mlflow.search_experiments()]
runs_df = mlflow.search_runs(experiment_ids=all_experiments)

# Filter hanya sesi 'Production' dan 'Transfer/Subject' (Abaikan data iterasi Optuna)
mask = runs_df['tags.mlflow.runName'].str.contains('Production|E9|E10', na=False, case=False)
prod_runs = runs_df[mask].copy()

columns_to_keep = [
    'tags.mlflow.runName',
    'params.F1', 'params.D', 'params.dropout_rate',
    'metrics.best_val_accuracy', 
    'metrics.final_word_accuracy',
    'metrics.base_val_accuracy',
    'metrics.finetune_val_accuracy'
]

# Ambil kolom yang eksis
existing_cols = [c for c in columns_to_keep if c in prod_runs.columns]
df_clean = prod_runs[existing_cols].copy()

# Rename Kolom untuk tabel
rename_map = {
    'tags.mlflow.runName': 'Eksperimen',
    'params.F1': 'Filter_Temporal',
    'params.D': 'Filter_Spasial',
    'metrics.best_val_accuracy': 'Akurasi_Suku_Kata',
    'metrics.final_word_accuracy': 'Akurasi_Kata',
}
df_clean.rename(columns=rename_map, inplace=True)

# Format persentase
if 'Akurasi_Suku_Kata' in df_clean.columns:
    df_clean['Akurasi_Suku_Kata'] = (df_clean['Akurasi_Suku_Kata'] * 100).round(2)
if 'Akurasi_Kata' in df_clean.columns:
    df_clean['Akurasi_Kata'] = (df_clean['Akurasi_Kata'] * 100).round(2)

# Tampilkan Tabel Master yang telah diurutkan
display(df_clean.sort_values(by='Eksperimen').reset_index(drop=True))

## 2. Analisis Tahap 1 & 2: Pemrosesan Sinyal & Eksplorasi Spasio-Temporal
Membandingkan **Baseline (E0)** dengan **ICA (E1)**, **Resampling (E2)**, **ERP Cropping N400 (E3)**, dan **Channel Ablation (E4)**.

In [ ]:
# Filter skenario E0 sampai E4
tahap_1_2 = df_clean[df_clean['Eksperimen'].str.contains('E0|E1|E2|E3|E4', na=False)]

if not tahap_1_2.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(
        data=tahap_1_2.sort_values('Eksperimen'), 
        x='Eksperimen', y='Akurasi_Suku_Kata', palette='crest'
    )
    plt.xticks(rotation=25, ha='right')
    plt.ylim(0, 100)
    plt.title('Dampak Teknik Preprocessing & Pemotongan Spasio-Temporal terhadap Akurasi EEGNet', fontweight='bold')
    plt.ylabel('Akurasi Validasi (%)')
    
    # Auto-label
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', fontweight='bold', xytext=(0, 5), textcoords='offset points')
    plt.tight_layout()
    plt.show()
else:
    print("⏳ Data Tahap 1 & 2 belum tersedia di database.")

## 3. Analisis Tahap 3 & 4: Augmentasi Data & Fitur
Membandingkan **Baseline (E0)** dengan **Augmentasi/Noise (E5)**, **Cross-Modality (E6)**, dan **Isolasi Pita Frekuensi Alpha (E7)**.

In [ ]:
# Filter skenario E0, E5, E6, E7
tahap_3_4 = df_clean[df_clean['Eksperimen'].str.contains('E0|E5|E6|E7', na=False)]

if not tahap_3_4.empty:
    plt.figure(figsize=(9, 5))
    ax = sns.barplot(
        data=tahap_3_4.sort_values('Eksperimen'), 
        x='Eksperimen', y='Akurasi_Suku_Kata', palette='magma'
    )
    plt.xticks(rotation=20, ha='right')
    plt.ylim(0, 100)
    plt.title('Efektivitas Augmentasi Sinyal dan Isolasi Pita Frekuensi', fontweight='bold')
    plt.ylabel('Akurasi Validasi (%)')
    
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', fontweight='bold', xytext=(0, 5), textcoords='offset points')
    plt.tight_layout()
    plt.show()

## 4. Galeri Kinerja Model (Confusion Matrix & SHAP)
Inspeksi visual metrik klasifikasi dan *Explainable AI* (SHAP) untuk memahami sensor otak mana (AF3, F7, dll.) yang paling berpengaruh saat memikirkan kata tertentu pada eksperimen pilihan.

In [ ]:
def view_experiment_reports(exp_id):
    """Memuat gambar output dari folder laporan spesifik"""
    reports_dir = os.path.abspath(os.path.join("..", "backend", "reports", exp_id))
    
    if not os.path.exists(reports_dir):
        print(f"❌ Folder laporan untuk '{exp_id}' belum dibuat (Eksperimen belum dijalankan/selesai).")
        return
        
    print(f"\n{'='*60}")
    print(f"📊 INSPEKSI VISUAL: {exp_id}")
    print(f"{'='*60}")
    
    cm_files = sorted([f for f in os.listdir(reports_dir) if f.startswith('cm_')])
    for cm in cm_files:
        print(f"\n➤ {cm}:")
        display(Image(filename=os.path.join(reports_dir, cm), width=500))
        
    shap_files = sorted([f for f in os.listdir(reports_dir) if f.startswith('shap_')])
    if shap_files:
        print(f"\n➤ Contoh Penjelasan AI (DeepLIFT SHAP Heatmap):")
        # Tampilkan 1-2 sampel pertama saja agar tidak memenuhi layar
        for shap_img in shap_files[:2]: 
            display(Image(filename=os.path.join(reports_dir, shap_img), width=700))

# Ubah ID di bawah ini untuk menginspeksi hasil eksperimen yang berbeda
# Contoh: "E0_Baseline", "E1_ICA_Filtering", "E3_ERP_N400"
view_experiment_reports("E0_Baseline")